# Iceberg tables in AIDP: Time Travel, Rollback, Data Retention 

This walkthrough uses a small orders table in an existing AIDP Spark session and expects the cells be run in order. Ensure valid permission to create, read, write, and manage the table. Re-run from setup to start with a fresh table. Individual write cells should run once.


### Set up a few orders

Starting point: **3 orders totaling amount to 600**.


In [62]:
import re
import time
import uuid

catalog = "default"
schema = "default"

# Avoid collision with existing tables. 
table = f"{catalog}.{schema}.orders_history_{uuid.uuid4().hex[:8]}"
system = f"{catalog}.system"


# Create an iceberg table. 
spark.sql(f"""
CREATE TABLE {table} (
    order_id INT,
    customer STRING,
    region STRING,
    amount DOUBLE,
    order_ts TIMESTAMP
)
USING ICEBERG
""")

spark.sql(f"""
INSERT INTO {table} VALUES
    (1, 'Alice', 'west', 100.0, TIMESTAMP '2026-01-31 00:00:00'),
    (2, 'Bob',   'east', 200.0, TIMESTAMP '2026-02-01 00:00:00'),
    (3, 'Cara',  'west', 300.0, TIMESTAMP '2026-02-02 00:00:00')
""")

# Helper to read the current checkpoint from the main reference.
def current_snapshot_id():
    return int(spark.sql(f"""
        SELECT snapshot_id FROM {table}.refs WHERE name = 'main'
    """).collect()[0]["snapshot_id"])

print("Demo table:", table)
spark.sql(f"SELECT * FROM {table} ORDER BY order_id").show()


Demo table: default.default.orders_history_39f6a3f2


+--------+--------+------+------+-------------------+
|order_id|customer|region|amount|           order_ts|
+--------+--------+------+------+-------------------+
|       1|   Alice|  west| 100.0|2026-01-31 00:00:00|
|       2|     Bob|  east| 200.0|2026-02-01 00:00:00|
|       3|    Cara|  west| 300.0|2026-02-02 00:00:00|
+--------+--------+------+------+-------------------+



## 1. Time travel: inspect a report's earlier input

Time travel is useful when investigating why a report changed or checking what data was available before a load. First capture the current snapshot and give it a readable name. A tag lets us refer to that checkpoint without copying its data into another table.


In [63]:
good_snapshot = current_snapshot_id()
spark.sql(f"""
ALTER TABLE {table}
CREATE TAG report_checkpoint AS OF VERSION {good_snapshot}
""").collect()

# Capture a commit-time marker before the next load.
good_time = spark.sql("""
SELECT date_format(current_timestamp(), 'yyyy-MM-dd HH:mm:ss.SSSSSS') AS ts
""").collect()[0]["ts"]

print("Report checkpoint:", good_snapshot)
print("As-of timestamp:", good_time)


Report checkpoint: 3431377720461754152
As-of timestamp: 2026-10-15 13:25:35.662926


Now suppose a load adds a spurious order worth 9,999. Today's table contains **4 orders totaling 10,599**. We will inspect the earlier state before deciding how to recover.


In [64]:
spark.sql(f"""
INSERT INTO {table} VALUES
    (99, 'IncorrectLoad', 'west', 9999.0,
     TIMESTAMP '2026-02-03 00:00:00')
""")

spark.sql(f"SELECT * FROM {table} ORDER BY order_id").show()


+--------+-------------+------+------+-------------------+
|order_id|     customer|region|amount|           order_ts|
+--------+-------------+------+------+-------------------+
|       1|        Alice|  west| 100.0|2026-01-31 00:00:00|
|       2|          Bob|  east| 200.0|2026-02-01 00:00:00|
|       3|         Cara|  west| 300.0|2026-02-02 00:00:00|
|      99|IncorrectLoad|  west|9999.0|2026-02-03 00:00:00|
+--------+-------------+------+------+-------------------+



Query the earlier state by timestamp and by tag. Both historical counts should be **3**, while the current count remains **4**. These reads do not repair or modify the current table. The `order_ts` values inside the rows do not determine the as-of checkpoint.


In [65]:
# Both queries use checkpoints captured before the incorrect load.
historical_count = spark.sql(f"""
SELECT COUNT(*) AS cnt
FROM {table} TIMESTAMP AS OF '{good_time}'
""").collect()[0]["cnt"]

tagged_count = spark.sql(f"""
SELECT COUNT(*) AS cnt
FROM {table} VERSION AS OF 'report_checkpoint'
""").collect()[0]["cnt"]

current_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {table}").collect()[0]["cnt"]
print("At timestamp:", historical_count)
print("At tagged checkpoint:", tagged_count)
print("Current:", current_count)
assert historical_count == tagged_count == 3
assert current_count == 4


At timestamp: 3
At tagged checkpoint: 3
Current: 4


## 2. Rollback: make the good checkpoint current again

When the load itself is wrong, rollback makes ordinary readers see the earlier good state again. Here, the saved snapshot is an ancestor of the current state: the only intervening write was the incorrect append. NOTE: This restores the whole checkpoint, so choose it before accepting any later valid writes.


In [66]:
# Restore the snapshot saved before order 99 was introduced.
spark.sql(f"""
CALL {system}.rollback_to_snapshot(
    table => '{table}',
    snapshot_id => {good_snapshot}
)
""").show(truncate=False)

assert current_snapshot_id() == good_snapshot
assert spark.sql(f"SELECT COUNT(*) AS cnt FROM {table} WHERE order_id = 99").collect()[0]["cnt"] == 0
spark.sql(f"SELECT * FROM {table} ORDER BY order_id").show()


+--------------------+-------------------+
|previous_snapshot_id|current_snapshot_id|
+--------------------+-------------------+
|8349343188755981770 |3431377720461754152|
+--------------------+-------------------+



+--------+--------+------+------+-------------------+
|order_id|customer|region|amount|           order_ts|
+--------+--------+------+------+-------------------+
|       1|   Alice|  west| 100.0|2026-01-31 00:00:00|
|       2|     Bob|  east| 200.0|2026-02-01 00:00:00|
|       3|    Cara|  west| 300.0|2026-02-02 00:00:00|
+--------+--------+------+------+-------------------+



The current table is back to **3 orders totaling 600**. Subsequent queries no longer need a historical clause to see the corrected data. NOTE: Rollback changes which checkpoint is current, it is different and separate from expiring historical snapshots.

## 3. Data retention: manage history separately from current rows

Retention has two decisions: **which historical checkpoints to keep**, and **which business records to keep in the current table**. A reporting checkpoint may need to stay available even when intermediate checkpoints can be retired.

Our `report_checkpoint` tag protects the good state. Add two valid orders and capture the checkpoint between them. That intermediate snapshot will be our expiration candidate, the newest snapshot must remain current.


In [67]:
spark.sql(f"""
INSERT INTO {table} VALUES
    (4, 'Dev', 'east', 40.0, TIMESTAMP '2026-02-03 00:00:00')
""")
expirable_snapshot = current_snapshot_id()

spark.sql(f"""
INSERT INTO {table} VALUES
    (5, 'Eve', 'west', 50.0, TIMESTAMP '2026-02-04 00:00:00')
""")
latest_snapshot = current_snapshot_id()

assert len({good_snapshot, expirable_snapshot, latest_snapshot}) == 3
spark.sql(f"SELECT * FROM {table} ORDER BY order_id").show()


+--------+--------+------+------+-------------------+
|order_id|customer|region|amount|           order_ts|
+--------+--------+------+------+-------------------+
|       1|   Alice|  west| 100.0|2026-01-31 00:00:00|
|       2|     Bob|  east| 200.0|2026-02-01 00:00:00|
|       3|    Cara|  west| 300.0|2026-02-02 00:00:00|
|       4|     Dev|  east|  40.0|2026-02-03 00:00:00|
|       5|     Eve|  west|  50.0|2026-02-04 00:00:00|
+--------+--------+------+------+-------------------+



### Retire an intermediate snapshot

The current table now contains **5 orders totaling 690**. Explicitly expire the untagged, non-current intermediate snapshot. This makes the effect observable immediately without waiting for an age threshold. Both, the tagged reporting checkpoint and the latest snapshot, must survive.


In [68]:
spark.sql(f"""
CALL {system}.expire_snapshots(
    table => '{table}',
    snapshot_ids => array({expirable_snapshot}),
    stream_results => true
)
""").show(truncate=False)

remaining = {
    int(row["snapshot_id"])
    for row in spark.sql(f"""
        SELECT snapshot_id FROM {table}.snapshots ORDER BY committed_at
    """).collect()
}
assert expirable_snapshot not in remaining
assert good_snapshot in remaining
assert latest_snapshot in remaining
assert current_snapshot_id() == latest_snapshot
assert spark.sql(f"SELECT COUNT(*) AS cnt FROM {table}").collect()[0]["cnt"] == 5
print("Intermediate snapshot expired; tagged and current snapshots retained.")


+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|deleted_data_files_count|deleted_position_delete_files_count|deleted_equality_delete_files_count|deleted_manifest_files_count|deleted_manifest_lists_count|deleted_statistics_files_count|
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|0                       |0                                  |0                                  |0                           |1                           |0                             |
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+



Intermediate snapshot expired; tagged and current snapshots retained.


Orders 4 and 5 remain in the table: expiring an intermediate checkpoint does not undo its rows from later states.

### Retire a business record

For this small example, January's only order is order 1. Delete that known record to check business-row retention.


In [69]:
spark.sql(f"DELETE FROM {table} WHERE order_id = 1")

assert spark.sql(f"SELECT COUNT(*) AS cnt FROM {table}").collect()[0]["cnt"] == 4
assert spark.sql(f"SELECT COUNT(*) AS cnt FROM {table} WHERE order_id = 1").collect()[0]["cnt"] == 0
spark.sql(f"SELECT * FROM {table} ORDER BY order_id").show()


+--------+--------+------+------+-------------------+
|order_id|customer|region|amount|           order_ts|
+--------+--------+------+------+-------------------+
|       2|     Bob|  east| 200.0|2026-02-01 00:00:00|
|       3|    Cara|  west| 300.0|2026-02-02 00:00:00|
|       4|     Dev|  east|  40.0|2026-02-03 00:00:00|
|       5|     Eve|  west|  50.0|2026-02-04 00:00:00|
+--------+--------+------+------+-------------------+



The current table contains **4 orders totaling 590**. Deleting a business row does not by itself erase its historical copies. Any retained checkpoint can keep older data accessible, so it needs to be ensured that reporting tags and historical retention follow the same policy as the business data.

| Operation | What changes | Current orders | Current total |
|---|---|---:|---:|
| Initial load | Starting data | 3 | 600 |
| Incorrect load | Current data | 4 | 10,599 |
| Time-travel reads | Nothing in the current table | 4 | 10,599 |
| Rollback | Current checkpoint | 3 | 600 |
| Two valid appends | Current data | 5 | 690 |
| Snapshot expiration | Available history | 5 | 690 |
| Business-row deletion | Current data | 4 | 590 |

The demo table and reporting tag remain available for inspection. No automated table cleanup is performed. 
